In [ ]:
# Cài đặt môi trường sạch
!pip install -q --upgrade diffusers transformers accelerate torch opencv-python rembg streamlit pyngrok onnxruntime-gpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 899.7/899.7 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 626.5 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 125.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 68.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 72.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 MB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.5/267.5 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 288.2/288.2 MB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.3/39.3 MB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.0/90.0 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17

In [ ]:
# Cài thư viện tải file từ Drive
!pip install -q gdown

# --- THAY ID DƯỚI ĐÂY BẰNG ID FILE LORA CỦA BẠN ---
file_id = '1qDWKACa-WhGMBEaYMXsYCGnwLXVrglVZ'
output_path = '/content/prostudio.safetensors' # Lưu thẳng vào thư mục gốc

import gdown
gdown.download(f'https://drive.google.com/uc?id={file_id}', output_path, quiet=False)

print("✅ Đã tải xong file LoRA!")

In [ ]:
%%writefile app.py
import streamlit as st
import torch
from diffusers import StableDiffusionControlNetInpaintPipeline, ControlNetModel, DDIMScheduler
from PIL import Image
import cv2
import numpy as np
from rembg import remove
import gc

# --- CẤU HÌNH ---
LORA_PATH = "/content/prostudio.safetensors"
TRIGGER_WORD = "prostudio style"

# Dùng Model DreamShaper 8 (Mới hơn, đẹp hơn, hỗ trợ safetensors chuẩn)
BASE_MODEL = "Lykon/dreamshaper-8-inpainting"

@st.cache_resource
def load_models():
    # 1. Load ControlNet
    st.write("⏳ Đang tải ControlNet (1/3)...")
    controlnet = ControlNetModel.from_pretrained(
        "lllyasviel/control_v11p_sd15_canny",
        torch_dtype=torch.float16,
        use_safetensors=True
    )

    # 2. Load Model Chính (Dùng DreamShaper)
    st.write(f"⏳ Đang tải Model chính: {BASE_MODEL} (2/3)...")
    pipe = StableDiffusionControlNetInpaintPipeline.from_pretrained(
        BASE_MODEL,
        controlnet=controlnet,
        torch_dtype=torch.float16,
        use_safetensors=True,
        safety_checker=None,
        low_cpu_mem_usage=True # <--- QUAN TRỌNG: Chống tràn RAM
    )

    # 3. Load LoRA
    st.write("⏳ Đang nạp LoRA (3/3)...")
    try:
        pipe.load_lora_weights(LORA_PATH)
        pipe.fuse_lora()
        st.write("✅ Đã nạp LoRA thành công!")
    except Exception as e:
        st.warning(f"⚠️ Không tìm thấy LoRA (Sẽ chạy model gốc). Lỗi: {e}")

    # 4. Đẩy sang GPU
    pipe.scheduler = DDIMScheduler.from_config(pipe.scheduler.config)
    pipe.to("cuda")

    return pipe

def process_image(input_image, prompt, negative_prompt, pipe):
    w, h = 512, 512

    # Xử lý ảnh đầu vào
    no_bg = remove(input_image)
    mask = Image.fromarray(255 - np.array(no_bg.split()[-1]))

    img_array = np.array(input_image.resize((w, h)))
    canny = cv2.Canny(img_array, 100, 200)
    canny = np.concatenate([canny[:, :, None]] * 3, axis=2)
    canny_img = Image.fromarray(canny)

    # Dọn dẹp bộ nhớ trước khi vẽ
    torch.cuda.empty_cache()
    gc.collect()

    result = pipe(
        prompt=f"{TRIGGER_WORD}, {prompt}, best quality, 8k, masterpiece, ultra realistic",
        negative_prompt=negative_prompt,
        image=input_image.resize((w, h)),
        mask_image=mask.resize((w, h)),
        control_image=canny_img.resize((w, h)),
        num_inference_steps=30,
        guidance_scale=7.5,
        strength=1.0
    ).images[0]

    return result

# --- GIAO DIỆN ---
st.set_page_config(layout="wide", page_title="AI Studio V2")
st.title("📸 AI Product Studio (DreamShaper V8)")

# Khu vực trạng thái (Status Container)
with st.status("Hệ thống đang khởi động...", expanded=True) as status:
    try:
        pipe = load_models()
        status.update(label="✅ Hệ thống đã sẵn sàng!", state="complete", expanded=False)
    except Exception as e:
        st.error(f"Lỗi nghiêm trọng: {e}")
        status.update(label="❌ Khởi động thất bại", state="error")
        st.stop()

col1, col2 = st.columns(2)
with col1:
    f = st.file_uploader("Upload ảnh gốc", type=["jpg", "png"])
    if f: st.image(Image.open(f), caption="Ảnh gốc")

with col2:
    p = st.text_area("Mô tả bối cảnh:", "on a wooden table, sunlight, shadows")
    if st.button("🚀 TẠO ẢNH", type="primary") and f:
        with st.spinner("Đang vẽ... (Mất khoảng 10-15s)"):
            res = process_image(Image.open(f).convert("RGB"), p, "ugly, bad quality", pipe)
            st.image(res, caption="Kết quả")

Writing app.py


In [ ]:
!pkill -f streamlit
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64
# Chạy app
print("🚀 Đang mở Web...")
!streamlit run app.py &>/dev/null &
import time; time.sleep(5)
!./cloudflared-linux-amd64 tunnel --url http://localhost:8501

🚀 Đang mở Web...
2025-12-12T07:15:12Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2025-12-12T07:15:12Z INF Requesting new quick Tunnel on trycloudflare.com...
2025-12-12T07:15:17Z INF +--------------------------------------------------------------------------------------------+
2025-12-12T07:15:17Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2025-12-12T07:15:17Z INF |  https://performing-likelihood-handhe